# Docker — Local Build & Smoke Testing

> **Notebook flow:** 01 Setup → 02 EDA → 03 Train → **[04 Docker]** → 05 Kubernetes → 06 Cleanup · 07 Azure Deploy · 08 API Tests

This notebook builds and tests the **dual-container architecture** locally: a training container that produces `model.pkl`, then an inference container that loads it at runtime via a mounted volume (not baked into the image).

**Prerequisites:**
- Dev Container is running with Docker-outside-of-Docker
- Docker Desktop is running on the host machine
- Training data exists at the path configured in `config.yaml data.raw_path` — run `01_devcontainer_setup.ipynb §6` to verify

---

## Flow
```
Verify Docker → Build train image → Run training → Verify artifact → Build infer image → Run infer container (model mounted) → Smoke test → Clean up
```


In [ ]:
import os
import time
import json
import subprocess
import yaml
import pandas as _pd

ROOT = "/workspaces/marketing-model-mlops-azure"
TRAIN_IMAGE_TAG = "bank-marketing-train:local"
INFER_IMAGE_TAG = "bank-marketing-api:local"

# Naming convention: bm-<purpose>  (matches kind cluster: bm-local, CI containers: bm-<stage>)
# Change the purpose suffix to scope containers per test session if needed.
CONTAINER_NAME  = "bm-smoke-test"
assert CONTAINER_NAME.startswith("bm-"), f"Container name must follow 'bm-<purpose>' convention, got: {CONTAINER_NAME!r}"

API_PORT = 8000

# Load config to derive artifact paths — avoids hardcoding paths that live in config.yaml
with open(os.path.join(ROOT, "config.yaml")) as f:
    _config = yaml.safe_load(f)

MODEL_PATH   = os.path.join(ROOT, _config["artifacts"]["model_path"])
METRICS_PATH = os.path.join(ROOT, _config["artifacts"]["metrics_path"])

# Auto-generate sample payload from the first row of the configured training CSV.
# This keeps SAMPLE_PAYLOAD dataset-agnostic — no bank-marketing-specific feature names hardcoded.
# If the CSV cannot be loaded (e.g. data not yet present), set SAMPLE_PAYLOAD manually below.
try:
    _raw_path = os.path.join(ROOT, _config["data"]["raw_path"])
    _sep      = _config["data"].get("separator", ",")
    _target   = _config["model"]["target_column"]
    _row      = _pd.read_csv(_raw_path, sep=_sep, nrows=1)
    _feature_cols = [c for c in _row.columns if c != _target]
    # Convert numpy scalars → Python native types so json.dumps works
    SAMPLE_PAYLOAD = {
        "features": {
            k: (v.item() if hasattr(v, "item") else v)
            for k, v in _row[_feature_cols].iloc[0].items()
        }
    }
    print(f"Sample payload auto-loaded from: {_config['data']['raw_path']}")
    print(f"Feature columns ({len(_feature_cols)}): {_feature_cols}")
except Exception as _e:
    print(f"⚠️  Could not auto-load sample payload: {_e}")
    print("   Set SAMPLE_PAYLOAD manually: SAMPLE_PAYLOAD = {'features': {'col': val, ...}}")
    SAMPLE_PAYLOAD = {"features": {}}

# Export to env so %%bash cells can reference these as $VAR instead of hardcoding strings
os.environ["TRAIN_IMAGE_TAG"]  = TRAIN_IMAGE_TAG
os.environ["INFER_IMAGE_TAG"]  = INFER_IMAGE_TAG
os.environ["CONTAINER_NAME"]   = CONTAINER_NAME
os.environ["PREDICT_PAYLOAD"]  = json.dumps(SAMPLE_PAYLOAD)

os.chdir(ROOT)
print(f"\nWorking directory:  {os.getcwd()}")
print(f"Train image tag:    {TRAIN_IMAGE_TAG}")
print(f"Infer image tag:    {INFER_IMAGE_TAG}")
print(f"Container name:     {CONTAINER_NAME}")
print(f"Model path:         {MODEL_PATH}")
print(f"Metrics path:       {METRICS_PATH}")
print(f"Payload features:   {list(SAMPLE_PAYLOAD['features'].keys())}")


## 1. Verify Docker Access

Confirm the Docker CLI can reach the host daemon via Docker-outside-of-Docker.

In [ ]:
%%bash
echo "=== Docker version ==="
docker --version

echo ""
echo "=== Docker daemon info ==="
docker info 2>&1 | grep -E 'Server Version|Operating System|Total Memory' || echo "ERROR: Cannot connect to Docker daemon — is Docker Desktop running on the host?"

## 2. Build the Training Image

Builds the training container from `Dockerfile.train`. This image runs `python main.py train` and writes `model.pkl` + `metrics.json` to a mounted `/app/artifacts` volume.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Building training image: $TRAIN_IMAGE_TAG"
docker build -f Dockerfile.train -t "$TRAIN_IMAGE_TAG" .

echo ""
echo "=== Image created ==="
docker images | grep bank-marketing-train


## 3. Run Training Container

Runs the training container with mounted volumes. Data is read from `data/` and artifacts are written to `artifacts/`.

    "> This mirrors the CI pipeline's `TrainModel` stage — the training container produces `model.pkl` which is then loaded by the inference container at startup (via volume mount locally, or Azure Blob Storage in production)."

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

# DooD: Docker daemon runs on the host — volume paths must resolve on the host,
# not inside the devcontainer.  LOCAL_WORKSPACE_FOLDER holds the host-side path
# (set by the Dev Container runtime).  Convert Windows backslashes → forward
# slashes and the drive letter to /c/ style so Docker Desktop can resolve it.
if [[ -n "$LOCAL_WORKSPACE_FOLDER" ]]; then
  HOST_DIR=$(echo "$LOCAL_WORKSPACE_FOLDER" | sed -E 's/^([a-zA-Z]):\\/\/\L\1\//; s/\\/\//g')
else
  HOST_DIR="$(pwd)"
fi

echo "Host workspace: $HOST_DIR"
echo "Running training container: $TRAIN_IMAGE_TAG"
docker run --rm \
  -v "${HOST_DIR}/data:/app/data" \
  -v "${HOST_DIR}/artifacts:/app/artifacts" \
  "$TRAIN_IMAGE_TAG"

echo ""
echo "=== Training complete ==="
echo "Artifacts:"
ls -lh artifacts/model.pkl artifacts/metrics.json 2>/dev/null || echo "ERROR: artifacts not found"


## 4. Verify Model Artifact

Confirm the training container produced `model.pkl`. This artifact is mounted into the inference container at runtime — it is **not** baked into the inference image.

In [ ]:
if os.path.exists(MODEL_PATH):
    size_kb = os.path.getsize(MODEL_PATH) / 1024
    print(f"OK  {_config['artifacts']['model_path']} found ({size_kb:.1f} KB)")
else:
    print(f"MISSING  {_config['artifacts']['model_path']}")
    print("Training container did not produce the expected artifact.")
    raise FileNotFoundError(f"{MODEL_PATH} not found — re-run Section 3.")


## 5. Validate Training Container Output

Verify the training container produced valid artifacts — not just that the files exist, but that the metrics are sensible and the model is loadable. This catches issues like corrupted serialisation, wrong sklearn version, or a broken pipeline.

In [ ]:
import joblib

# --- Validate metrics.json ---
with open(METRICS_PATH) as f:
    metrics = json.load(f)

print("=== Metrics validation ===")
required_keys = ["model_type", "test_size", "roc_auc", "f1_minority_class", "f1_macro"]
for key in required_keys:
    assert key in metrics, f"FAIL: missing key '{key}' in metrics.json"
    print(f"  {key}: {metrics[key]}")

assert (
    0.0 < metrics["roc_auc"] <= 1.0
), f"FAIL: roc_auc out of range: {metrics['roc_auc']}"
assert (
    0.0 < metrics["f1_macro"] <= 1.0
), f"FAIL: f1_macro out of range: {metrics['f1_macro']}"
print("PASS: metrics.json valid")

# --- Validate model.pkl is loadable ---
print("\n=== Model validation ===")
pipeline = joblib.load(MODEL_PATH)
assert hasattr(pipeline, "predict"), "FAIL: loaded object has no predict method"
assert hasattr(
    pipeline, "predict_proba"
), "FAIL: loaded object has no predict_proba method"
print(f"  Pipeline steps: {[name for name, _ in pipeline.steps]}")
print(f"  Model type:     {type(pipeline.named_steps['classifier']).__name__}")
print("PASS: model.pkl loadable and has expected interface")


## 6. Training Container Reproducibility Check

Re-run the training container and confirm the metrics are identical. This validates that the container is deterministic (fixed random seed from `config.yaml`) and produces consistent artifacts across runs.

> This mirrors what CI does: if someone rebuilds the training image, the output must be identical given the same data and config.

In [ ]:
# Save first-run metrics for comparison
with open(METRICS_PATH) as f:
    metrics_before = json.load(f)

# DooD: Resolve host-side workspace path for volume mounts
host_dir = os.environ.get("LOCAL_WORKSPACE_FOLDER", ROOT)
# Convert Windows paths if needed (e.g. C:\Users\... → /c/Users/...)
if "\\" in host_dir:
    import re

    host_dir = re.sub(r"^([a-zA-Z]):\\", lambda m: f"/{m.group(1).lower()}/", host_dir)
    host_dir = host_dir.replace("\\", "/")

# Re-run training container
result = subprocess.run(
    [
        "docker",
        "run",
        "--rm",
        "-v",
        f"{host_dir}/data:/app/data",
        "-v",
        f"{host_dir}/artifacts:/app/artifacts",
        TRAIN_IMAGE_TAG,
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(f"FAIL: training container exited with code {result.returncode}")
    print(result.stderr)
    raise RuntimeError("Training container failed on second run")

# Compare metrics
with open(METRICS_PATH) as f:
    metrics_after = json.load(f)

print("=== Reproducibility check ===")
all_match = True
for key in ["roc_auc", "f1_minority_class", "f1_macro"]:
    before, after = metrics_before[key], metrics_after[key]
    status = "MATCH" if before == after else "MISMATCH"
    if before != after:
        all_match = False
    print(f"  {key}: {before} → {after}  [{status}]")

if all_match:
    print("PASS: training container is reproducible (deterministic output)")
else:
    print("WARN: metrics differ between runs — check random_state in config.yaml")


## 7. Build the Inference Image

Builds the inference container from `Dockerfile.infer`. The image does **not** contain `model.pkl` — the model is loaded at runtime via a mounted volume (local) or Azure Blob Storage (AKS).

> `model.pkl` must exist in `artifacts/` on the host. When running the container locally, the `artifacts/` directory is mounted as a read-only volume.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Building inference image: $INFER_IMAGE_TAG"
docker build -f Dockerfile.infer -t "$INFER_IMAGE_TAG" .

echo ""
echo "=== Image created ==="
docker images | grep bank-marketing-api


## 8. Run the Inference Container

Start the inference API container in the background with the model.pkl mounted as a volume, and wait for it to finish loading the model.

> **Volume mount:** The `artifacts/` directory is mounted read-only into the container at `/app/artifacts`. This means retraining only requires restarting the container — no image rebuild needed.

> **DooD networking note:** In Docker-outside-of-Docker, `docker run -p 8000:8000` publishes the port to the *host machine's* localhost — not to `localhost` inside the devcontainer. All `curl` commands in this notebook run inside the devcontainer, so they would get "connection refused".
>
> Fix: `--network container:$(hostname)` shares the devcontainer's network namespace with the API container. Port 8000 then resolves correctly at `localhost:8000` from inside the devcontainer.

In [ ]:
%%bash
# Stop and remove any pre-existing container with the same name
docker rm -f "$CONTAINER_NAME" 2>/dev/null || true

# DooD: Docker daemon runs on the host — volume paths must resolve on the host,
# not inside the devcontainer.  LOCAL_WORKSPACE_FOLDER holds the host-side path.
if [[ -n "$LOCAL_WORKSPACE_FOLDER" ]]; then
  HOST_DIR=$(echo "$LOCAL_WORKSPACE_FOLDER" | sed -E 's/^([a-zA-Z]):\\/\/\L\1\//; s/\\/\//g')
else
  HOST_DIR="$(pwd)"
fi

echo "Starting inference container '$CONTAINER_NAME' (model mounted from $HOST_DIR/artifacts)..."
docker run -d \
  --name "$CONTAINER_NAME" \
  --network container:$(hostname) \
  -v "${HOST_DIR}/artifacts:/app/artifacts:ro" \
  "$INFER_IMAGE_TAG"

# --network container:$(hostname) shares the devcontainer's network namespace.
# -v .../artifacts:/app/artifacts:ro mounts model.pkl at runtime (not baked in).

echo "Waiting 5 seconds for model to load..."
sleep 5

echo ""
echo "=== Container status ==="
docker ps --filter "name=$CONTAINER_NAME" --format 'table {{.Names}}\t{{.Status}}'


## 9. Health Check

Confirm the API is alive and the model loaded correctly.

In [ ]:
%%bash
echo "=== Health check ==="
curl -sf http://localhost:8000/health | python3 -m json.tool

echo ""
echo "Expected: {\"status\": \"healthy\"}"

## 10. Prediction Smoke Test

Send a sample payload to `/predict` and verify the response structure.

In [ ]:
%%bash
echo "=== Prediction request ==="
curl -sf -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d "$PREDICT_PAYLOAD" | python3 -m json.tool

echo ""
echo "Expected: {\"prediction\": ..., \"probability\": ..., \"label\": \"yes\" or \"no\"}"


## 11. Edge Case Tests

Validate API behaviour for boundary inputs and validation errors.

| Scenario | Expected |
|---|---|
| Young customer (age 18) | Valid prediction |
| Negative balance | Valid prediction (signed-log handles it) |
| Missing `features` key entirely | HTTP 422 |
| Empty `features` dict | HTTP 422 |


In [ ]:
%%bash
BASE_URL="http://localhost:8000"

echo "--- Young customer (age 18) ---"
curl -sf -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"features":{"age":18,"job":"student","marital":"single","education":"secondary","default":"no","balance":100.0,"housing":"no","loan":"no","contact":"cellular","day":5,"month":"jan","duration":60.0,"campaign":1,"pdays":-1,"previous":0,"poutcome":"unknown"}}' \
  | python3 -m json.tool

echo ""
echo "--- Negative balance ---"
curl -sf -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"features":{"age":45,"job":"blue-collar","marital":"married","education":"primary","default":"no","balance":-500.0,"housing":"yes","loan":"yes","contact":"telephone","day":10,"month":"jun","duration":120.0,"campaign":3,"pdays":-1,"previous":0,"poutcome":"unknown"}}' \
  | python3 -m json.tool

echo ""
echo "--- Missing features key entirely — expect HTTP 422 ---"
curl -s -o /dev/null -w "%{http_code}" -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{}'
echo " (expected: 422)"

echo ""
echo "--- Empty features dict — expect HTTP 422 ---"
curl -s -o /dev/null -w "%{http_code}" -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"features":{}}'
echo " (expected: 422)"


## 12. Scripted Pass/Fail Smoke Test

Replicates the CI smoke test logic: runs health check and prediction, exits non-zero on failure.

In [ ]:
%%bash
set -e

echo "=== Smoke test ==="

HEALTH=$(curl -sf http://localhost:8000/health)
echo "$HEALTH" | grep -q '"healthy"' || { echo "FAIL: health check"; exit 1; }
echo "PASS: health check"

PRED=$(curl -sf -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d "$PREDICT_PAYLOAD")
echo "$PRED" | grep -q '"prediction"' || { echo "FAIL: predict endpoint"; exit 1; }
echo "PASS: predict endpoint"

echo ""
echo "All smoke tests passed."
echo "Response: $PRED"


## 13. View Container Logs & Clean Up

Inspect API startup logs, then stop and remove the container started in this notebook.

> **Full environment teardown** (images, kind cluster, artifacts, binaries): run **`06_cleanup.ipynb`** instead.


In [ ]:
%%bash
echo "=== Container logs ==="
docker logs "$CONTAINER_NAME"

echo ""
echo "=== Stopping and removing container ==="
docker stop "$CONTAINER_NAME" && docker rm "$CONTAINER_NAME"
echo "Clean up complete."


---

## Summary

| Step | Section | Expected result |
|---|---|---|
| Verify Docker | §1 | Daemon reachable |
| Build train image | §2 | Image created |
| Run training container | §3 | `model.pkl` + `metrics.json` produced |
| Verify artifact | §4 | File present, correct size |
| Validate training output | §5 | Metrics valid, model loadable with predict interface |
| Reproducibility check | §6 | Metrics identical across two runs |
| Build infer image | §7 | Image created (no model baked in) |
| Run infer container | §8 | Container running with model mounted as volume |
| Health check | §9 | `{"status":"healthy"}` |
| Predict | §10 | `{"prediction":...}` |
| Edge cases | §11 | HTTP 422 for invalid input |
| Pass/fail smoke test | §12 | All checks pass |
| Logs + clean up | §13 | Container removed |

### Model Loading Modes

The inference container loads `model.pkl` at startup. The source depends on the environment:

| Environment | How model.pkl is loaded | Config |
|---|---|---|
| **Local Docker** | Volume mount (`-v ./artifacts:/app/artifacts:ro`) | Default (`STORAGE_BACKEND=local`) |
| **AKS (production)** | Azure Blob Storage at startup | `STORAGE_BACKEND=azure_blob` |
| **Override** | Custom path via env var | `MODEL_PATH=/custom/path/model.pkl` |

Once the Docker smoke test passes, open **`05_kubernetes_setup.ipynb`** to deploy to the local kind cluster.